# Simulation Calibration: Can the Digital Twin Correct Itself?

*Use case 2 of 3, about 20 minutes.*

Like many state-of-the-art simulators in the field, OpenDC uses a deterministic and static simulation strategy.
To stay accurate, OpenDT adds a Self-Calibrator, which measures the difference between simulation-predicted results and
actual telemetry, then continuously adjusts the simulation model.

In [ ]:
import numpy as np
import pandas as pd
from resources import *

check_setup()

## 1. The Power Model

We model CPU power draw adopting the OpenDC analytical formula $P(u) = P_{idle} + (P_{max} - P_{idle})(2u - u^r)$.
Here, $u$ is CPU utilization, $P_{idle}$ and $P_{max}$ represent idle and max power, and $r$ is the calibration
parameter.
For the SURF hosts, OpenDT uses $P_{idle}$ = 25 W, $P_{max}$ = 174 W, and $r$ = 4.

In [ ]:
u = np.linspace(0, 1, 101)  # CPU utilization, from 0 (idle) to 1 (full load)
P_IDLE, P_MAX = 25, 174  # watts

curves = {f"r = {r}": P_IDLE + (P_MAX - P_IDLE) * (2 * u - u**r) for r in [2, 4, 6, 9]}
ax = pd.DataFrame(curves, index=u * 100).plot()
ax.axhline(P_MAX, color="black", linewidth=0.8, linestyle="--", label="$P_{max}$")
ax.set(xlabel="CPU Utilization [%]", ylabel="Power Draw [W]", xlim=(0, 100), ylim=(0, None))
legend_above(ax)

*Question:* in notebook 1, OpenDT under-estimates the ground truth in almost every minute; should the Self-Calibrator
increase or decrease r to close the gap?

<details><summary>Answer</summary>

Increase r: at the same CPU utilization, a higher r gives a higher power draw (e.g., at 50% utilization,
165 W with r = 4 vs. 174 W with r = 9).
</details>

*Question (optional):* for r > 2, the curves exceed $P_{max}$ (174 W) before full load; can a SURF host draw more than
its maximum power?

<details><summary>Answer</summary>

No; the formula is an empirical model (Fan et al., ISCA 2007), and with r > 2 it exceeds $P_{max}$ below
full load (e.g., 202 W at 80% utilization with r = 4).
</details>

## 2. Calibrating r with a Grid Search

The Self-Calibrator employs a grid search strategy over the power model's parameter space: it evaluates a set of
candidate parameter values by running short simulations and comparing the results against recent historical telemetry
data. The configuration yielding the lowest MAPE is selected and transmitted to the simulator for use in subsequent
predictions.
Here, we run one simulation of the full week per r, from 3 to 9, as in experiment E2 of the OpenDT paper (one to two
minutes).

In [ ]:
R_VALUES = [3, 4, 5, 6, 7, 8, 9]
sweep = simulate_many("r", R_VALUES, until=TRACE_END)  # one simulation of the full week per r


def error(run):
    """Error of a simulation at each minute [%]: |ground truth - prediction| / ground truth."""
    both = align(run)  # ground truth and prediction on the same 1-minute grid
    return (both["measured"] - both["simulated"]).abs() / both["measured"] * 100


errors = pd.DataFrame({r: error(run) for r, run in sweep.items()})  # one column per r
pd.DataFrame({"MAPE [%]": errors.mean()}).round(1)  # one row per r

*Question:* which r has the lowest MAPE over the week, and is it the r = 4 of the SURF topology?

<details><summary>Answer</summary>

r = 7, with a MAPE of 4.5% (vs. 6.4% for r = 4).
</details>

## 3. The Best r per Day

In [ ]:
best_r_per_day = errors.resample("24h", origin=TRACE_START).mean().idxmin(axis=1)
best_r_per_day

*Question:* does the best r stay the same from one day to the next, and what does this mean for a digital twin that
runs for weeks?

<details><summary>Answer</summary>

No; the best r changes between 5 and 9 from day to day.
Static simulation models can drift and introduce errors as the time horizon increases: hardware behavior varies with
temperature, aging, and firmware updates, while workload characteristics evolve over time.
Therefore, OpenDT recalibrates continuously.
</details>

## 4. The Self-Calibrating Digital Twin

In experiment E2, OpenDT twins one hour at a time; after each hour, the Self-Calibrator picks the r with the lowest MAPE
over the last hour, and the next hour uses it (in the live twin, the new r may arrive later).
OpenDC is deterministic, so we replay this from the simulations above, without new simulations.
The plot also shows the 10% error rate OpenDT should stay within (NFR1, community-accepted) and the MAPE of the
peer-reviewed simulator FootPrinter (7.86%).

*Question:* the Self-Calibrator only sees the last hour, while r = 7 is the best r for the whole week in hindsight;
which of the two has the lower MAPE?

In [ ]:
hourly = errors.resample("1h").mean()  # MAPE of each r, per hour


def self_calibrate(hourly, picked):
    """MAPE per hour when each hour uses the r picked after the previous hour (r = 4 first)."""
    used = picked.shift(1, fill_value=4)
    return pd.Series([hourly.loc[hour, r] for hour, r in used.items()], index=hourly.index)


best_r = hourly.idxmin(axis=1)  # the r with the lowest MAPE in each hour
no_calibration = hourly[4]
with_calibration = self_calibrate(hourly, best_r)

for name, mape_per_hour in [("No Calibration", no_calibration), ("With Calibration", with_calibration)]:
    print(f"{name}: MAPE {mape_per_hour.mean():.1f}%, hours within 10%: {(mape_per_hour < 10).mean():.0%}")

In [ ]:
both = pd.DataFrame({"No Calibration": no_calibration, "With Calibration": with_calibration})
ax = both.plot(color=["#E69F00", "#009E73"], style=["--", "-"])
ax.axhline(10, color="red", linewidth=0.8, linestyle=":", label="NFR Threshold (10%)")
ax.axhline(7.86, color="#338ec1", linewidth=0.8, linestyle="-.", label="FootPrinter (7.86%)")
ax.set(xlabel="Time [UTC]", ylabel="MAPE [%]", ylim=(0, None))
legend_above(ax)

*Question:* in the OpenDT paper, calibration improves the MAPE from 5.13% to 4.39%, and the time within 10% from 86% to
92%; how close are the numbers here, and why do they differ?

<details><summary>Answer</summary>

Same trend; the self-calibrating twin even beats the best fixed r (3.6% vs. 4.5%).
The numbers differ because of the method: the paper computes the MAPE on 5-minute averages, after aligning the two
series in time, with 10 values of r; here, 1-minute samples (as in the Self-Calibrator) and 7 values of r.
As in the paper, calibration alleviates underestimation bias, reducing the frequency of underestimated predictions
(here, from 93% to 69% of the minutes; in the paper, from 85% to 66%).

![MAPE over time in the OpenDT paper](resources/images/paper_mape_over_time.png)

*MAPE over time with and without calibration, SURF week (OpenDT paper, Figure 6).*
</details>

*Question:* the calibrated twin has the lower MAPE over the week; is it also better in every single hour?

In [ ]:
worse = with_calibration > no_calibration
print(f"Calibration is worse than no calibration in {worse.sum()} of {len(worse)} hours.")

<details><summary>Answer</summary>

No; in 5 of 168 hours, the r picked over the last hour does not fit the next one.
It would be tempting to conclude that using calibration is always beneficial; however, there exist significant
simulation-periods when the no-calibration technique performs better than the OpenDT calibration.
This suggests calibration is important but its technique requires further attention.
</details>

*Exercise 1:* the default `mape_window_minutes` in the code of OpenDT is 1,440 (24 hours), while experiment E2 uses 60;
the next cell replays the self-calibrating twin with the 24-hour window, so which window gives the lower MAPE?

In [ ]:
last_24h = errors.rolling("24h").mean().resample("1h").last()  # MAPE of each r over the last 24 hours
with_24h = self_calibrate(hourly, last_24h.idxmin(axis=1))
print(f"MAPE {with_24h.mean():.1f}%, worse than no calibration in {(with_24h > no_calibration).sum()} hours")

<details><summary>Answer</summary>

Worse: MAPE 4.3% (vs. 3.6%), and worse than no calibration in 20 hours (vs. 5); a longer window reacts later
to drift.
</details>

*Exercise 2 (optional):* in how many of the 168 hours does each r win (next cell), and is the grid from 3 to 9 wide
enough?

In [ ]:
best_r.value_counts().sort_index()

<details><summary>Answer</summary>

r = 9, the edge of the grid, in 75 of 168 hours; the grid may be too narrow.
</details>

*Question:* with three more simulations of the full week (r = 10, 11, and 12), does the self-calibrating twin improve,
and where is the best r now?

In [ ]:
sweep_wide = simulate_many("r", R_VALUES + [10, 11, 12], until=TRACE_END)  # r = 3 to 9 come from the cache
errors_wide = pd.DataFrame({r: error(run) for r, run in sweep_wide.items()})
hourly_wide = errors_wide.resample("1h").mean()
best_r_wide = hourly_wide.idxmin(axis=1)
with_wide = self_calibrate(hourly_wide, best_r_wide)
print(f"r from 3 to 12: MAPE {with_wide.mean():.1f}%")
best_r_wide.value_counts().sort_index()

<details><summary>Answer</summary>

A little: MAPE 3.3% (vs. 3.6%), but now r = 12 is the best in 69 hours, again at the edge; in these hours,
OpenDT under-estimates the power draw even with r = 12.
</details>

## Calibration in the Live Twin

*Optional, requires Docker.* The same grid search, in `resources/configs/live_twin_calibration.yaml`:

```yaml
calibrator:
  calibrated_property: "cpuPowerModel.calibrationFactor"  # r in the power model
  min_value: 3
  max_value: 9
  linspace_points: 7  # r = 3, 4, ..., 9
  max_parallel_workers: 4
  mape_window_minutes: 60
```

Start it with `make up config=tutorial/resources/configs/live_twin_calibration.yaml` and follow it with
`make logs-calibrator`: after each grid search, the MAPE of each candidate, the best value
(`Best: cpuPowerModel.calibrationFactor=...`), and `Published topology with ...`. More in notebook 4.

*Next:* [3.co2_migration.ipynb](3.co2_migration.ipynb), CO2-aware workload migration.